# **Exploración de los datos**
---

## **1. Librerías**
---

In [ ]:
import pandas as pd
import os
import sys
import argparse 
from datetime import datetime
from dateutil import parser
import matplotlib.pyplot as plt
import numpy as np


## **2. Análisis de la base de datos**
---

### **2.1 Cargue de la base de datos**
---

In [ ]:
# Leer solo las primeras N filas (aprox 50%)
df_tranx = pd.read_csv(
    ruta_archivo,
    nrows=90_000_000,  # ~50% de 180M
    low_memory=False
)

In [ ]:
ruta_data_original = 'C:\\Users\\cecor\\OneDrive - Universidad Nacional de Colombia\\GitHub\\MSc-Thesis-Financial-Fraud-Detection-Models\\data\\original'
archivo_tranx = 'HI-Large_Trans.csv'
ruta_archivo = os.path.join(ruta_data_original, archivo_tranx)
#
if os.path.exists(ruta_data_original):
    # low_memory=False para inferir los tipos de datos.
    df_tranx = pd.read_csv(ruta_archivo, low_memory=False, nrows=20_000_000)
    #Vista rápida
    print(f"Dimensiones del dataset: {df_tranx.shape}")
    print("\nPrimeras 5 filas:")
    display(df_tranx.head())
    #
    print("\nInformación de columnas y tipos de datos:")
    print(df_tranx.info())

else:
    print("Error: No se encuentra el archivo")
    print(f"Ruta intentada: {os.path.abspath(ruta_data_original)}")

Dimensiones del dataset: (20000000, 11)

Primeras 5 filas:


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/08/01 00:17,20,800104D70,20,800104D70,6794.63,US Dollar,6794.63,US Dollar,Reinvestment,0
1,2022/08/01 00:02,3196,800107150,3196,800107150,7739.29,US Dollar,7739.29,US Dollar,Reinvestment,0
2,2022/08/01 00:17,1208,80010E430,1208,80010E430,1880.23,US Dollar,1880.23,US Dollar,Reinvestment,0
3,2022/08/01 00:03,1208,80010E650,20,80010E6F0,73966883.00,US Dollar,73966883.00,US Dollar,Cheque,0
4,2022/08/01 00:02,1208,80010E650,20,80010EA30,45868454.00,US Dollar,45868454.00,US Dollar,Cheque,0



Información de columnas y tipos de datos:
<class 'pandas.DataFrame'>
RangeIndex: 20000000 entries, 0 to 19999999
Data columns (total 11 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Timestamp           str    
 1   From Bank           int64  
 2   Account             str    
 3   To Bank             int64  
 4   Account.1           str    
 5   Amount Received     float64
 6   Receiving Currency  str    
 7   Amount Paid         float64
 8   Payment Currency    str    
 9   Payment Format      str    
 10  Is Laundering       int64  
dtypes: float64(2), int64(3), str(6)
memory usage: 2.7 GB
None


In [4]:
path='C:\\Users\\cecor\\OneDrive - Universidad Nacional de Colombia\\GitHub\\MSc-Thesis-Financial-Fraud-Detection-Models\\data\\original'
os.chdir(path)

In [ ]:
muestra = df_tranx.sample(200000, random_state=42)
muestra.to_csv('muestra_tranx.csv', index=False)

: 

### **2.2 Análisis de la base de datos**
---

In [ ]:
# Cambio el nombre de las columnas
df_tranx.columns=df_tranx.columns.str.strip().str.lower().str.replace(' ', '_')
df_tranx.columns

Index(['bank_name', 'bank_id', 'account_number', 'entity_id', 'entity_name'], dtype='str')

In [4]:
# Verificar valores nulos
for i in df_accounts.columns:
    print(f"El número de valores nulos para la columna: {i}")
    print(f" es de: {df_accounts[i].isnull().sum()}")

El número de valores nulos para la columna: bank_name
 es de: 0
El número de valores nulos para la columna: bank_id
 es de: 0
El número de valores nulos para la columna: account_number
 es de: 0
El número de valores nulos para la columna: entity_id
 es de: 0
El número de valores nulos para la columna: entity_name
 es de: 0


In [5]:
# Mirar la distribución de las variables
for i in df_accounts.columns:
    if df_accounts[i].dtype == 'object':
        print(f"Columna: {i}")
        print(df_accounts[i].nunique())

In [6]:
print(f"Dado que el número de filas del conjunto de datos es de:{df_accounts.shape[0]:,}.") 
print(f"y el número de cuentas únicas es de: {df_accounts['account_number'].nunique():,}.  Se concluye que hay: {df_accounts.shape[0] - df_accounts['account_number'].nunique():,} cuentas repetidas.")

Dado que el número de filas del conjunto de datos es de:2,126,855.
y el número de cuentas únicas es de: 2,110,228.  Se concluye que hay: 16,627 cuentas repetidas.


In [7]:
# Validación de duplicados en account_key
duplicados_cuenta=df_accounts['account_number'].value_counts()
print(duplicados_cuenta[duplicados_cuenta>1].head(2))

account_number
82EE0ED30    4
8322DBCF0    4
Name: count, dtype: int64


In [8]:
# Creo una llave unica para cada cuenta combinando bank_id y account_number
# No hay valores duplicados después de crear la llave única, lo que es una buena señal.
df_accounts["account_key"]=df_accounts["bank_id"].astype(str)+"-"+df_accounts["account_number"].astype(str)
duplicados_cuenta_key=df_accounts['account_key'].value_counts()
if duplicados_cuenta_key[duplicados_cuenta_key>1].empty:
    print("No hay cuentas duplicadas después de crear la llave única.")
else:
    print("Hay cuentas duplicadas después de crear la llave única:")
    print(duplicados_cuenta_key[duplicados_cuenta_key>1])

No hay cuentas duplicadas después de crear la llave única.


In [9]:
# tipo de corporacion
df_accounts['entity_type'] = df_accounts['entity_name'].str.split('#').str[0].str.strip()
df_accounts["entity_type"].value_counts()

entity_type
Partnership            745341
Corporation            692680
Sole Proprietorship    680013
Country                  5689
Individual               3104
Direct                     28
Name: count, dtype: int64

In [10]:
# Las entidades pueden tener más de 1 banco adscrito
cuentas_por_entidad = df_accounts.groupby('entity_id')['bank_id'].nunique().reset_index()
cuentas_por_entidad.columns = ['entity_id', 'num_bancos_distintos']
cuentas_por_entidad.head(3)

,entity_id,num_bancos_distintos
0,2AA02E7E570,2
1,2AA02E7E5F0,15
2,2AA02E7E650,2


In [11]:
# Unir esto al DataFrame original
df_accounts = df_accounts.merge(cuentas_por_entidad, on='entity_id', how='left')

In [12]:
pd.concat([df_accounts.head(2), df_accounts.tail(2)])
df_accounts.head()

,bank_name,bank_id,account_number,entity_id,entity_name,account_key,entity_type,num_bancos_distintos
0,Portugal Bank #500,240522,82655C500,2AA04EEC5D0,Corporation #82502,240522-82655C500,Corporation,10
1,First Bank of Laramie,339367,8505ED380,2AA06B8AC80,Partnership #15630,339367-8505ED380,Partnership,21
2,National Bank of Helena,368763,826283D80,2AA066499F0,Corporation #12918,368763-826283D80,Corporation,18
3,Switzerland Bank #2454,3174937,842090C80,2AA06712690,Corporation #135403,3174937-842090C80,Corporation,477
4,China Bank #561,53267,817D00980,2AA053417D0,Corporation #103595,53267-817D00980,Corporation,15


### **2.3 Exportar la base de datos**
---

In [13]:
ruta_data_limpio = 'C:\\Users\\cecor\\OneDrive - Universidad Nacional de Colombia\\GitHub\\MSc-Thesis-Financial-Fraud-Detection-Models\\data\\processed'
archivo_cuentas = 'HI-Large_accounts_limpio'
ruta_archivo = os.path.join(ruta_data_limpio, archivo_cuentas + '.parquet')
#
try:
    # index=False para no columna extra
    df_accounts.to_parquet(ruta_archivo, index=False, engine='pyarrow')
    print(f"¡Archivo exportado exitosamente en:\n{ruta_archivo}")    
except Exception as e:
    print(f"Ocurrió un error al exportar: {e}")

¡Archivo exportado exitosamente en:
C:\Users\cecor\OneDrive - Universidad Nacional de Colombia\GitHub\MSc-Thesis-Financial-Fraud-Detection-Models\data\processed\HI-Large_accounts_limpio.parquet
